In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import defaultdict, deque
from copy import deepcopy
from tqdm import tqdm
import pandas as pd
import torch
import json
import ast
import re

In [ ]:
model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
def normalize_output(output_str):
    try:
        data = json.loads(output_str.replace("'", '"'))
    except Exception:
        return output_str

    def remove_bbox(obj):
        if isinstance(obj, dict):
            return {k: remove_bbox(v) for k, v in obj.items() if k != "bbox_2d"}
        elif isinstance(obj, list):
            return [remove_bbox(x) for x in obj]
        else:
            return obj

    clean_data = remove_bbox(deepcopy(data))
    return repr(clean_data)

def group_dataframe(df):
    df = df.copy()
    df["output_clean"] = df["output"].apply(normalize_output)

    grouped = (
        df.groupby(["input", "input_ita", "output_clean"], sort=False, group_keys=False)
          .agg({
              "id": lambda x: list(x),
              "image_path": lambda x: list(x),
              "output": lambda x: list(x)
          })
          .reset_index()
    )
    return grouped

In [ ]:
def replace_surfaces(mapping_str, frames_str):
    mapping = ast.literal_eval(mapping_str)
    frames = ast.literal_eval(frames_str)

    mapping_dict = defaultdict(deque)
    for m in mapping:
        if len(m) == 2:
            orig, trans = m
            if trans == "":
                mapping_dict[orig].append({"remove": True})
            else:
                mapping_dict[orig].append({"surface": trans})
        elif len(m) == 3:
            orig, trans, ref = m
            if trans == "":
                mapping_dict[orig].append({"remove": True})
            else:
                mapping_dict[orig].append({"surface": trans, "reference": ref})

    for frame in frames:
        new_elements = []
        for el in frame["elements"]:
            surf = el["surface"]
            if mapping_dict[surf]:
                replacement = mapping_dict[surf].popleft()
                if "remove" in replacement:
                    continue
                el.update(replacement)
            new_elements.append(el)
        frame["elements"] = new_elements

    return str(frames)

In [ ]:
def align_frame(en_text, ita_text, frame):
    system_content = """
    TASK DESCRIPTION
    Extract lexical mappings from a structured JSON by aligning each extracted English surface form to its best-matching SINGLE token in the provided Italian sentence. 
    Produce pairs for ordinary tokens and triples for third‑person pronouns, preserving the exact traversal order of the JSON.
    
    INPUT
    You are given three items:
    1) English sentence: <EN_TEXT>
    2) Italian sentence: <IT_TEXT>
    3) ArrayJSON: <ARRAY_JSON>
    - ArrayJSON is a list of frames; each frame has an "elements" list; each element has a "surface" field.
    - Only the "surface" field is eligible for extraction; ignore all other fields (e.g. frame, name).
    
    RULES
    - Extraction:
      - Traverse the ArrayJSON in order; within each frame, process elements in their listed order.
      - For each element, extract the exact string in its "surface" field; call it surface_extracted.
    - Mapping to Italian:
      - IMPORTANT: Extract ONLY ONE token from <IT_TEXT> (e.g., "sala" for "living room" and NOT "sala da pranzo").
      - For ordinary tokens, return the SINGLE Italian token from <IT_TEXT> that best matches surface_extracted in meaning and role; (e.g., "sacchetti" for "bags", "cassetto" for "drawer").
      - Operational state keywords: if surface_extracted is "on" or "off" referring to device state, return it verbatim as the Italian token ("<ON>" or "<OFF>"), regardless of its presence in <IT_TEXT>.
      - Missing matches: if no suitable Italian token for surface_extracted is present in <IT_TEXT>, set italian_word to '' (empty string). Do NOT invent or infer elided/implied tokens.
      - Pronouns: if surface_extracted is a third‑person pronoun (e.g., "it", "they", "her", "this"...):
        - italian_word = the Italian pronoun token exactly as it appears in <IT_TEXT>; if it is a clitic attached to a verb (e.g., "portarlo", "cercarla", "prendili"), extract only the clitic itself ("lo", "la", "li" etc.). If no pronoun token is present in <IT_TEXT>, set italian_word to ''.
        - reference = the SINGLE Italian token in <IT_TEXT> that is the antecedent of the pronoun, chosen by nearest suitable context alignment with <EN_TEXT>. If no antecedent token is present in <IT_TEXT>, set reference to ''.
    - Do not introduce any new surfaces: only process surfaces that appear in the JSON. Keep duplicates if they recur.
    - Maintain order strictly identical to the JSON traversal order.
    - No commentary, no reasoning, no extra tokens.
    
    OUTPUT
    - Produce a single list:
      - For ordinary tokens and operational state keywords: output a pair (surface_extracted, italian_word).
      - For third‑person pronouns: output a triple (surface_extracted, italian_word, reference).
    - Use parentheses and single quotes exactly, comma‑separated, in a single bracketed list.
    - Output only the list, nothing else.
    
    EXAMPLES
    INPUT:
    English text: "take the red book on the dining table"
    Italian text: "prendi il libro rosso sul tavolo da pranzo"
    ArrayJSON: "[{'frame': 'TAKING', 'elements': [{'name': 'Theme', 'surface': 'book'}, {'name': 'Source', 'surface': 'table'}]}]"
    OUTPUT: "[('book','libro'), ('table','tavolo')]"

    INPUT:
    English text: "can you go in the kitchen"
    Italian text: "puoi andare in cucina"
    ArrayJSON: "[{'frame': 'MOTION', 'elements': [{'name': 'Theme', 'surface': 'you'}, {'name': 'Goal', 'surface': 'kitchen'}]}]"
    OUTPUT: "[('you',''), ('kitchen','cucina')]"

    INPUT:
    English text: "can you bring me the apple"
    Italian text: "puoi portarmi"
    ArrayJSON: "[{'frame': 'BRINGING', 'elements': [{'name': 'Agent', 'surface': 'you'}, {'name': 'Beneficiary', 'surface': 'me'}, {'name': 'Theme', 'surface': 'apple'}]}]"
    OUTPUT: "[('you',''), ('me','mi', ''), ('apple','mela')]"

    INPUT:
    English text: "bring me the book"
    Italian text: "portami il libro"
    ArrayJSON: "[{'frame': 'BRINGING', 'elements': [{'name': 'Beneficiary', 'surface': 'me'}, {'name': 'Theme', 'surface': 'book'}]}]"
    OUTPUT: "[('me','mi', ''), ('apple','libro')]"
    
    INPUT:
    English text: "take a fork and bring it here"
    Italian text: "prendi una forchetta e portala qui"
    ArrayJSON: "[{'frame': 'TAKING', 'elements': [{'name': 'Theme', 'surface': 'fork'}]}, {'frame': 'BRINGING', 'elements': [{'name': 'Theme', 'surface': 'it'}, {'name': 'Goal', 'surface': 'here'}]}]"
    OUTPUT: "[('fork','forchetta'), ('it','la','forchetta'), ('here','qui')]"

    INPUT:
    English text: "find the bottles in the fridge and put them on the table"
    Italian text: "trova le bottiglie nel frigorifero e mettile sul tavolo"
    ArrayJSON: "[{'frame': 'LOCATING', 'elements': [{'name': 'Theme', 'surface': 'bottles'}, {'name': 'Location', 'surface': 'fridge'}]}, {'frame': 'PLACING', 'elements': [{'name': 'Theme', 'surface': 'them'}, {'name': 'Goal', 'surface': 'table'}]}]"
    OUTPUT: "[('bottles','bottiglie'), ('fridge','frigorifero'), ('them','le','bottiglie'), ('table','tavolo')]"

    INPUT:
    English text: "turn off the fan then turn on the heater"
    Italian text: "spegni il ventilatore poi accendi il riscaldamento"
    ArrayJSON: "[{'frame': 'CHANGE_OPERATIONAL_STATE', 'elements': [{'name': 'Operational_state', 'surface': 'off'}, {'name': 'Device', 'surface': 'fan'}]}, {'frame': 'CHANGE_OPERATIONAL_STATE', 'elements': [{'name': 'Operational_state', 'surface': 'on'}, {'name': 'Device', 'surface': 'heater'}]}]"
    OUTPUT:"[('off','<OFF>'), ('fan','ventilatore'), ('on','<ON>'), ('heater','riscaldamento')]"

    INPUT:
    English text: "pick up the apples and the knife wash them and put it on the plate"
    Italian text: "prendi le mele e il coltello lavale e mettilo sul piatto"
    ArrayJSON: "[{'frame': 'PICKING_UP', 'elements': [{'name': 'Theme', 'surface': 'apples'}, {'name': 'CoTheme', 'surface': 'knife'}]}, {'frame': 'WASHING', 'elements': [{'name': 'Theme', 'surface': 'them'}]}, {'frame': 'PLACING', 'elements': [{'name': 'Theme', 'surface': 'it'}, {'name': 'Goal', 'surface': 'plate'}]}]"
    OUTPUT: "[('apples','mele'), ('knife','coltello'), ('them','le','mele'), ('it','lo','coltello'), ('plate','piatto')]"
    """

    user_content = f"English text: {en_text}\nItalian text: {ita_text}\nArrayJSON: {frame}"
    
    prompt = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": f"/no_think {user_content}"}
    ]
    
    model_inputs = tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        padding=True,
        enable_thinking=False
    ).to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=124,
        eos_token_id=tokenizer.eos_token_id
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    
    try:
        # index finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    
    return tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

In [ ]:
def process_file(df, output_file, save_interval=50):
    df_new = df.copy()
    df_new["output_ita"] = None

    grouped_df = group_dataframe(df)

    i = 0
    for _, row in tqdm(grouped_df.iterrows(), total=len(grouped_df), desc="Processando"):
        i += 1
        
        input_str = str(row["input"])
        input_ita_str = str(row["input_ita"])
        output_cleaned = str(row["output_clean"])

        try:
            result = align_frame(input_str, input_ita_str, output_cleaned)
    
            for id_val, img_val in zip(row["id"], row["image_path"]):
                mask = (df_new["id"] == id_val) & (df_new["image_path"] == img_val)
                if mask.any():
                    output_str = df_new.loc[mask, "output"].iloc[0]
                    output_ita = replace_surfaces(result, output_str)
                    df_new.loc[mask, "output_ita"] = str(output_ita)
    
        except Exception as e:
            print("Errore: ", e)
            print("Riga erroe: ", row["id"][0])
        finally:
            if (i == save_interval) and i > 0:
                df_new.to_csv(output_file, sep="\t", index=False)
                print(f"File parziale salvato a riga {row['id'][0]}")
                i = 0
            
    df_new.to_csv(output_file, sep='\t', index=False)
    print(f"File finale salvato in: {output_file}")

    return df_new

In [ ]:
%%time
input_file = "/path/to/input.tsv"
output_file = "/path/to/output.tsv"
df = pd.read_csv(input_file, sep='\t')
result = process_file(df, output_file)